## Time to get your hands dirty. Your first neural network; pick your favourite.

*(exam guidelines available [here](https://github.com/dgerosa/astrostatistics_bicocca_2026/blob/main/exams.md))*


For the last coding assignment, you'll need to implement a neural network. We'll look at a relatively simple binary classification problem. Here below are three options; completing one of them for the exam is enough

### Tasks:

1. Remember: scale your data appropriately

2. Decide on a testing strategy (a simple test/train split? a CV strategy? set a test set aside to be looked at at the very end?)

2. Decide your optimization metric.

3. Write down your network architecture. You can start from a fully connected, multi-layer perceptron (and then explore)

4. Use one the package among those we've seen. These include Tensorflow via keras, pytorch, and the MPL classifier implemented in scikit-learn. This is an opportunity to pick the one you're most interested in learning. 

5. Optimize the hyperparameters of your network. Explore different hyperparameters and see what fits the data best.  Do your best now to optimize the network architecture. Be creative!

6. Report on the perfomance of the network on the test set; report other metrics that have not been optimized.


### A few tips:

- In scikit-learn, remember that you can utilize all availables cores on your machine with `n_jobs=-1`. Print out the classification score for the training data, and the best parameters obtained by the cross validation.
- If it takes too long, run the hyperparameter optimization on a subset of the training set. Then retrain the full network using the best hyperparameters only.
- On cross validation, for scikit learn we've seen how to use `GridSearchCV` already. For Tensorflow, there's a really cool tool called [Tensorboard](https://www.tensorflow.org/tensorboard)

### Datasets:

You can choose one of these three problems:

- **1. Galaxies vs quasars (but with neural networks)** Go back to our SDSS data we've used in Lecture 19. We had color differences, and the task was to classifty quasars vs galaxies. Repeat that task with a neural network.

- **2. Can a computer learn if we're going to detect gravitational waves? (but with neural networks)** Go back to the SNR classifier for gravitational wave events, same data we've used in Lecture. We had properties of black hole binaries, and the task was to classify. Repeat that task with a neural network.

- **3. The HiggsML challenge** Branching out of astrophysics, let's mess around with a dataset of simulated but  realistic events from the ATLAS particle detector at CERN.
    - Data are at `solutions/higgs.tar.gz` (you need to uncompress with `tar -czvf`)
    - There are $N_{\rm samples} = 2.5\times 10^5$ entries with $N_{\rm features}=30$ features each. 
    - The taks is that of classifying these features against a set of labels, which are either `s` (source) or `b` (background).
    - For some info on both the physics and the dataset see [this document](https://higgsml.lal.in2p3.fr/files/2014/04/documentation_v1.8.pdf); includes a description of the features and how data have been padded (-999) for missing values.
    - This dataset was part of a challenge that run on Kaggle in 2014: https://higgsml.ijclab.in2p3.fr/ 




In [2]:
import pandas as pd 
import tensorflow
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from keras.callbacks import EarlyStopping, TensorBoard, ReduceLROnPlateau

I0000 00:00:1782812673.828164    1483 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782812673.848523    1483 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782812675.307580    1483 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782812678.979557    1483 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [3]:
df = pd.read_csv('/home/matti/uni/astrostatistics_bicocca_2026/solutions/higgs.csv')


In [4]:
df_filt = df.loc[:, ~df.columns.isin(['EventId','Label', 'KaggleSet', 'KaggleWeight', 'Weight'])]

In [5]:
y = df['Label']

In [6]:
y = np.array([1 if label == 's' else 0 for label in y], dtype='int32')

In [7]:
y

array([1, 0, 0, ..., 1, 0, 0], shape=(250000,), dtype=int32)

In [7]:
df_filt

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,-0.277,258.733,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497
1,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,125.157,...,-1.916,164.546,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226
2,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,197.814,...,-2.186,260.414,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251
3,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,75.968,...,0.060,86.062,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
4,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,57.983,...,-0.871,53.131,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,-999.000,71.989,36.548,5.042,-999.00,-999.000,-999.000,1.392,5.042,55.892,...,2.859,144.665,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000
249996,-999.000,58.179,68.083,22.439,-999.00,-999.000,-999.000,2.585,22.439,50.618,...,-0.867,80.408,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
249997,105.457,60.526,75.839,39.757,-999.00,-999.000,-999.000,2.390,22.183,120.462,...,-2.890,198.907,1,41.992,1.800,-0.166,-999.000,-999.00,-999.000,41.992
249998,94.951,19.362,68.812,13.504,-999.00,-999.000,-999.000,3.365,13.504,55.859,...,0.811,112.718,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000


In [8]:
df_filt_nan = df_filt.replace(-999, np.nan)


In [9]:
def rescale(d_train, d_test):
    imputer = SimpleImputer(strategy = 'median', add_indicator= True)
    scaler = StandardScaler()
    d_train_i = imputer.fit_transform(d_train)
    d_train_si = scaler.fit_transform(d_train_i)
    d_test_i = imputer.transform(d_test)
    d_test_si = scaler.transform(d_test_i)
    return d_train_si, d_test_si

I try different models with different number of layers/fuctions/neurons in a cv scheme

In [10]:
from sklearn.model_selection import train_test_split, KFold
from keras import layers

In [11]:
def build(m):
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [18]:
# keras.backend.clear_session()
def model_sel(select):
    if (select == 0):
    #model1 = 2 layer, all relu with differrent dropouts to prevent overfitting
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(32, activation='relu', ),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(32, activation='relu', ),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ])
        # model1.summary()
    
    elif (select == 1):
        model = keras.Sequential([
            layers.Input(shape = (41, )),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
        # model2.summary()
    elif (select == 2):
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.3),
            layers.Dense(64, activation='tanh'),
            layers.Dropout(0.2),
            layers.Dense(1, activation='sigmoid')
        ])
        # model3.summary()
    elif (select == 3):
        model = keras.Sequential([
            layers.Input(shape = (41, )),
            layers.Dense(128, activation='relu'),
            # layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(64, activation='relu'),
            layers.Dense(32, activation= 'relu'),
            layers.Dense(1, activation='sigmoid')
        ])
        # model4.summary()
    elif (select == 4):
    #equal to model 1 but without dropout and batchnormalization
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(128, activation='relu', ),
            layers.Dense(64, activation='relu', ),
            layers.Dense(1, activation='sigmoid')
        ])
    elif (select == 5):
        model = keras.Sequential([
            layers.Input(shape = (41, )),
            layers.Dense(128, activation='relu'),
            # layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(64, activation='relu'),
            layers.Dense(32, activation= 'relu'),
            layers.Dense(1, activation='sigmoid')
        ])
        # model4.summary()
    elif (select == 6):
    #equal to model 1 but without dropout and batchnormalization
        model = keras.Sequential([
            layers.Input(shape = (41,)),
            layers.Dense(128, activation='relu', ),
            layers.Dense(64, activation='relu', ),
            layers.Dense(1, activation='sigmoid')
        ])
    else: print('error!!!!! --> choose a model')
    return build(model)

In [12]:
data_train, data_test, y_train, y_test = train_test_split(df_filt_nan, y, test_size= 0.15, shuffle= True)
# prova1, prova2, y1, y2 = train_test_split(data_train, y_train, test_size= 0.2)

In [13]:
data_train

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
49687,104.419,89.848,73.536,19.674,NaN,NaN,NaN,3.088,19.674,75.686,...,2.164,136.033,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000
220863,97.240,1.360,68.087,150.940,3.680,451.065,-2.326,1.144,56.788,370.990,...,-0.723,403.103,3,91.138,-2.869,-2.877,57.979,0.811,2.495,252.559
141792,NaN,73.694,44.948,39.171,NaN,NaN,NaN,1.453,39.171,61.338,...,-2.560,144.949,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000
230472,126.696,33.120,97.006,229.389,NaN,NaN,NaN,1.200,3.182,454.236,...,1.729,444.260,1,229.658,0.718,-1.378,NaN,NaN,NaN,229.658
227638,85.884,1.468,57.258,312.582,0.507,281.309,0.734,0.654,26.701,625.047,...,-1.332,680.889,2,377.437,0.640,1.682,51.187,1.147,-1.959,428.625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90707,NaN,76.850,50.318,1.283,NaN,NaN,NaN,1.676,1.283,53.617,...,1.533,45.553,0,NaN,NaN,NaN,NaN,NaN,NaN,-0.000
96185,165.791,72.214,134.768,32.545,NaN,NaN,NaN,3.301,2.136,115.349,...,-3.130,166.141,1,30.720,-0.039,2.401,NaN,NaN,NaN,30.720
171803,183.141,66.538,149.638,4.616,NaN,NaN,NaN,4.023,4.616,58.280,...,2.950,185.307,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000
240186,NaN,99.384,69.861,26.917,NaN,NaN,NaN,2.859,26.917,71.148,...,-0.229,79.171,0,NaN,NaN,NaN,NaN,NaN,NaN,0.000


they are all so similar 

more epochs

In [16]:
kfold = KFold(shuffle = True, n_splits = 5)

acc = []
acc_std = []
loss = []
loss_std = []
for selec in [0, 1, 2, 3, 4, 5, 6]:
    acc_cv = []
    loss_cv = []

    fold_num = 1

    for (train_index, cv_test_index) in kfold.split(data_train):
        keras.backend.clear_session()
        data_cv_train, data_cv_test = data_train.iloc[train_index], data_train.iloc[cv_test_index]
        y_cv_train, y_cv_test = y_train[train_index], y_train[cv_test_index]
        d_cv_train, d_cv_test = rescale(data_cv_train, data_cv_test)

        early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,  #number of epochs done without improving --> stop!!
        restore_best_weights=True)

        log_directory = f"logs/model_{selec}/fold_{fold_num}"
        tensorboard_callback = TensorBoard(log_dir=log_directory, histogram_freq=1)

        model = model_sel(selec)
        model.fit(d_cv_train, y_cv_train, epochs = 50, callbacks = [early_stopping, tensorboard_callback], verbose = 1, validation_data=(d_cv_test, y_cv_test), batch_size=512)
        
        l, accuracy = model.evaluate(d_cv_test, y_cv_test)
        acc_cv.append(accuracy)
        loss_cv.append(l)
    
    fold_num +=1
    
    
    print(f'============================Done {model} =================================')

    acc.append(np.mean(acc_cv))
    acc_std.append(np.std(acc_cv))
    loss.append(np.mean(loss_cv))
    loss_std.append(np.std(loss_cv))

Epoch 1/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 417us/step - accuracy: 0.6951 - loss: 0.5915 - val_accuracy: 0.7810 - val_loss: 0.4586
Epoch 2/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7671 - loss: 0.4829 - val_accuracy: 0.8111 - val_loss: 0.4172
Epoch 3/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7923 - loss: 0.4503 - val_accuracy: 0.8174 - val_loss: 0.4038
Epoch 4/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8031 - loss: 0.4356 - val_accuracy: 0.8206 - val_loss: 0.3984
Epoch 5/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8074 - loss: 0.4286 - val_accuracy: 0.8235 - val_loss: 0.3919
Epoch 6/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8113 - loss: 0.4217 - val_accuracy: 0.8252 - val_loss: 0.3878
Epoch 7/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8138 - loss: 0.4167 - val_accuracy: 0.8255 - val_loss: 0.3849
Epoch 8/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8156 - loss: 0.4126 - val_accuracy: 

In [17]:
print(acc, '\n', acc_std)
print(loss, '\n', loss_std)

[np.float64(0.8352658987045288), np.float64(0.8404188275337219), np.float64(0.839590585231781), np.float64(0.8408658981323243), np.float64(0.8408329367637635), np.float64(0.841143524646759), np.float64(0.8413270592689515)] 
 [np.float64(0.002065951461164621), np.float64(0.0005326984389446946), np.float64(0.0006934729107900759), np.float64(0.0016383923456241313), np.float64(0.0013718767774478246), np.float64(0.0020535479604943305), np.float64(0.002412648605059949)]
[np.float64(0.36946567296981814), np.float64(0.3593297600746155), np.float64(0.3601659297943115), np.float64(0.35665799379348756), np.float64(0.35656089782714845), np.float64(0.35690987706184385), np.float64(0.3574362635612488)] 
 [np.float64(0.002570517145642334), np.float64(0.0025379907852665527), np.float64(0.001678873801575206), np.float64(0.002245043688192197), np.float64(0.0029205294439673614), np.float64(0.002707381133494732), np.float64(0.0039652879564766146)]


now redo but with reducing rate of learning when near the bottom

In [ ]:
kfold = KFold(shuffle = True, n_splits = 5)

acc = []
acc_std = []
loss = []
loss_std = []
for selec in [0, 1, 2, 3, 4, 5, 6]:
    acc_cv = []
    loss_cv = []

    fold_num = 1

    for (train_index, cv_test_index) in kfold.split(data_train):
        keras.backend.clear_session()
        data_cv_train, data_cv_test = data_train.iloc[train_index], data_train.iloc[cv_test_index]
        y_cv_train, y_cv_test = y_train[train_index], y_train[cv_test_index]
        d_cv_train, d_cv_test = rescale(data_cv_train, data_cv_test)

        early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,  #number of epochs done without improving --> stop!!
        restore_best_weights=True)

        lr_scheduler = ReduceLROnPlateau(
            monitor='val_loss', 
            factor=0.2,   # Shrink learning rate by 80% when stuck
            patience=3,   # Wait 3 epochs before shrinking
            min_lr=1e-6,
            verbose=1
        )

        log_directory = f"logs/model_{selec}/fold_{fold_num}"
        tensorboard_callback = TensorBoard(log_dir=log_directory, histogram_freq=1)

        model = model_sel(selec)
        model.fit(d_cv_train, y_cv_train, epochs = 50, callbacks = [early_stopping, tensorboard_callback, lr_scheduler], verbose = 0, validation_data=(d_cv_test, y_cv_test), batch_size=512)
        
        l, accuracy = model.evaluate(d_cv_test, y_cv_test)
        acc_cv.append(accuracy)
        loss_cv.append(l)
    
    fold_num +=1
    
    
    print(f'============================Done {model} =================================')

    acc.append(np.mean(acc_cv))
    acc_std.append(np.std(acc_cv))
    loss.append(np.mean(loss_cv))
    loss_std.append(np.std(loss_cv))


Epoch 20: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 24: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.

Epoch 28: ReduceLROnPlateau reducing learning rate to 8.000000525498762e-06.
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8371 - loss: 0.3680

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 27: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.

Epoch 31: ReduceLROnPlateau reducing learning rate to 8.000000525498762e-06.

Epoch 36: ReduceLROnPlateau reducing learning rate to 1.6000001778593287e-06.
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8368 - loss: 0.3668

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8327 - loss: 0.3731

Epoch 23: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.

Epoch 27: ReduceLROnPlateau reducing learning rate to 4.000

In [20]:
print(acc, '\n', acc_std)
print(loss, '\n', loss_std)

[np.float64(0.8351717591285706), np.float64(0.8399341225624084), np.float64(0.8393552899360657), np.float64(0.8420376539230346), np.float64(0.8422776341438294), np.float64(0.8417552828788757), np.float64(0.841637647151947)] 
 [np.float64(0.0016484896675957286), np.float64(0.0014967649638157779), np.float64(0.0024643096899366382), np.float64(0.0019542364183033475), np.float64(0.0006632440241702268), np.float64(0.0020625221497907664), np.float64(0.0009266605896638302)]
[np.float64(0.37008326053619384), np.float64(0.36127901673316953), np.float64(0.3612178683280945), np.float64(0.35413414239883423), np.float64(0.35490469336509706), np.float64(0.3549617111682892), np.float64(0.35545021295547485)] 
 [np.float64(0.0029736046409348923), np.float64(0.0024327508631095643), np.float64(0.004443753588813192), np.float64(0.0028303645582701733), np.float64(0.002131739050474932), np.float64(0.0028804727040584867), np.float64(0.001539887969326952)]


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs

the best is model 3, not train on all dataset and test over test set

In [21]:
acc_cv = []
loss_cv = []


keras.backend.clear_session()

early_stopping = EarlyStopping(
monitor='val_loss',
patience=5,  #number of epochs done without improving --> stop!!
restore_best_weights=True)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,   # Shrink learning rate by 80% when stuck
    patience=3,   # Wait 3 epochs before shrinking
    min_lr=1e-6,
    verbose=1
)
data_t, data_te = rescale(data_train, data_test)

model = model_sel(3)
model.fit(data_t, y_train, epochs = 50, callbacks = [early_stopping, lr_scheduler], verbose = 1, validation_data=(data_te, y_test), batch_size=512)

l, accuracy = model.evaluate(data_te, y_test)
acc_cv.append(accuracy)
loss_cv.append(l)

Epoch 1/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.8122 - loss: 0.4126 - val_accuracy: 0.8329 - val_loss: 0.3703 - learning_rate: 0.0010
Epoch 2/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8336 - loss: 0.3750 - val_accuracy: 0.8349 - val_loss: 0.3773 - learning_rate: 0.0010
Epoch 3/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - accuracy: 0.8366 - loss: 0.3681 - val_accuracy: 0.8363 - val_loss: 0.3638 - learning_rate: 0.0010
Epoch 4/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.8384 - loss: 0.3634 - val_accuracy: 0.8411 - val_loss: 0.3577 - learning_rate: 0.0010
Epoch 5/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.8402 - loss: 0.3611 - val_accuracy: 0.8414 - val_loss: 0.3573 - learning_rate: 0.0010
Epoch 6/50
416/416 ━━━━━━━━━━━━━━━━━━━━ -2s -5466us/step - accuracy: 0.8411 - loss: 0.3587 - val_accuracy: 0.8397 - val_loss: 0.3591 - learning_rate: 0.0010
Epoch 7/50
416/416 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8417 - los

In [22]:
print(f'accuravy {acc_cv[0]} \n loss {loss_cv[0]}')

accuravy 0.8425066471099854 
 loss 0.35212960839271545
